# SQLite Básico
### Fundamentos de Gestión de Datos · TECSUP · Semana 2

**Docente:** Pilar Rocío Sayán Mejía · **Motor:** SQLite · **Entorno:** Google Colab (Python)

| | |
|---|---|
| **Nivel** | Básico (1 de 3: Básico → Intermedio → Avanzado) |
| **Base de datos** | AynyTech Cusco: 2 tablas (`departamentos` y `empleados`) |
| **Sentencias que aprenderás** | `CREATE TABLE`, `INSERT`, `SELECT`, `WHERE`, `ORDER BY`, `LIMIT`, `DISTINCT`, `COUNT`, `SUM`, `AVG`, `MAX`, `MIN` |
| **Tiempo estimado** | 100 minutos |

## Cómo trabajar este cuaderno

1. Ejecuta las celdas **en orden**, de arriba hacia abajo (botón ▶ o `Shift + Enter`).
2. **No necesitas escribir código nuevo**: lee la consulta, ejecútala e interpreta el resultado.
3. Si algo falla, usa **Entorno de ejecución ▸ Reiniciar y ejecutar todo**: el cuaderno puede ejecutarse completo cuantas veces quieras sin duplicar datos.
4. Al final hay **ejercicios** con su solución escondida: inténtalos antes de mirarla.

## 1. ¿Qué es SQLite?

SQLite es un **motor de base de datos relacional**: guarda tablas (filas y columnas) y las consulta con el lenguaje estándar **SQL** (*Structured Query Language*). Ese lenguaje se creó en IBM en 1974 (Donald Chamberlin y Raymond Boyce) a partir de la teoría relacional de Edgar Codd, y se estandarizó en 1986.

### ¿Qué puedes hacer con SQLite?

- Crear tablas, insertar datos, consultarlos, actualizarlos y eliminarlos (**CRUD**: *Create, Read, Update, Delete*).
- Escribir lo que quieres obtener (**declarativo**): tú dices *qué* quieres y el motor decide *cómo* obtenerlo.

### Los cuatro grupos de sentencias

| Grupo | Nombre | Para qué sirve | Ejemplos |
|---|---|---|---|
| **DQL** | Data Query Language | Consultar datos | `SELECT` |
| **DML** | Data Manipulation Language | Cambiar datos | `INSERT`, `UPDATE`, `DELETE` |
| **DDL** | Data Definition Language | Definir la estructura | `CREATE TABLE`, `DROP TABLE` |
| **DCL** | Data Control Language | Dar o quitar permisos | `GRANT`, `REVOKE` (no existen en SQLite, que es de un solo usuario) |

En este nivel usarás sobre todo **DQL** (`SELECT`) y verás un poco de **DDL** y **DML** para crear y llenar la base.

### Comparativo de motores de bases de datos relacionales

| Motor | ¿Quién lo usa? | ¿Para qué? | ¿Es gratuito? |
|---|---|---|---|
| **SQLite** | Apps móviles, Colab, cursos | Archivos locales, sin servidor | Sí |
| MySQL | WordPress, apps web | Servidores web de tráfico medio | Sí (edición comunitaria) |
| PostgreSQL | Startups, ciencia de datos | Alto rendimiento, datos complejos | Sí |
| SQL Server | Empresas grandes, bancos | Sistemas corporativos, reportes | No (Microsoft) |
| Oracle Database | Grandes corporaciones | Misiones críticas, alto volumen | No (Oracle) |

### ¿Por qué SQLite en este curso?

- Guarda **toda la base de datos en un solo archivo** (`.db`).
- **No hay que instalar ni configurar un servidor**.
- Python lo trae incluido (módulo `sqlite3`), así que funciona en Google Colab sin instalar nada.
- Es el motor más usado en aplicaciones móviles (Android e iOS).

## 2. Preparación del entorno

La celda siguiente hace tres cosas, y **solo debes ejecutarla**:

1. Carga las dos librerías que usaremos: `sqlite3` (el motor de SQLite, que Python ya incluye) y `pandas` (para ver los resultados como tabla).
2. Abre (o crea) el archivo de la base de datos: `conn` es la **conexión**, como abrir la puerta del archivo.
3. Activa las **claves foráneas** (`PRAGMA foreign_keys = ON`). En SQLite vienen apagadas por defecto: sin esta línea, las relaciones entre tablas no se verifican.

A partir de aquí solo usaremos dos instrucciones:

- `conn.executescript("""...""")` para sentencias que **cambian** la base (`CREATE`, `INSERT`, `UPDATE`...).
- `pd.read_sql_query("""...""", conn)` para consultas `SELECT`: muestra el resultado como una tabla.

> Cada vez que abras Colab de nuevo debes volver a ejecutar esta celda.

In [1]:
# CELDA DE PREPARACIÓN: ejecútala una sola vez al inicio.
import sqlite3
import pandas as pd

conn = sqlite3.connect("aynitech_basico.db")      # crea (o abre) el archivo de la base de datos
conn.execute("PRAGMA foreign_keys = ON")   # SQLite necesita esta línea para respetar las claves foráneas

print("Versión de SQLite:", sqlite3.sqlite_version)

Versión de SQLite: 3.45.1


## 3. Nuestra base de datos ficticia: AynyTech Cusco

**AynyTech** es una empresa de tecnología ficticia ubicada en Cusco. En este nivel trabajamos con **2 tablas relacionadas**:

| Tabla | ¿Qué guarda? | Filas |
|---|---|---|
| `departamentos` | Las áreas de la empresa (Sistemas, Ventas...) | 5 |
| `empleados` | Los trabajadores y el departamento donde trabajan | 8 |

```
departamentos  ◄──────  empleados
 (id_depto)              (id_depto)      Un departamento tiene MUCHOS empleados.
                                         Cada empleado pertenece a UN departamento.
```

La flecha indica que `empleados` tiene una **clave foránea** (`id_depto`) que apunta a la clave primaria de `departamentos`.

### Tipos de datos en SQLite

| Tipo | ¿Qué guarda? | Ejemplo |
|---|---|---|
| `INTEGER` | Números enteros | `1`, `42`, `-5` |
| `REAL` | Números decimales | `3.14`, `1500.50` |
| `TEXT` | Texto (y también **fechas**, como `'2024-05-10'`) | `'Cusco'`, `'Maria Lopez'` |
| `NUMERIC` | Número si el texto se puede convertir (afinidad numérica) | `1200.00` |
| `BLOB` | Datos binarios (imágenes, archivos) | Poco usado en cursos |

> En SQLite **no existe un tipo `DATE`**: las fechas se guardan como `TEXT` con formato `'AAAA-MM-DD'`.

### Las 4 restricciones fundamentales

| Restricción | ¿Qué significa? | Ejemplo |
|---|---|---|
| `PRIMARY KEY` | Identifica cada fila de forma **única**. No se repite ni puede ser nula | `id_emp INTEGER PRIMARY KEY` |
| `FOREIGN KEY` | Apunta a la clave primaria de **otra tabla**; crea la relación | `FOREIGN KEY (id_depto) REFERENCES departamentos(id_depto)` |
| `UNIQUE` | El valor no puede repetirse en la columna, pero **sí puede ser nulo** | `email TEXT UNIQUE` |
| `NOT NULL` | El campo es **obligatorio** | `nombre TEXT NOT NULL` |

### Crear las tablas

Primero se crea la tabla **padre** (la que no tiene clave foránea) y después la **hija**. Al principio, `DROP TABLE IF EXISTS` borra versiones anteriores (en orden inverso: primero la hija) para que puedas volver a ejecutar la celda sin errores.

In [2]:
conn.executescript("""
DROP TABLE IF EXISTS empleados;
DROP TABLE IF EXISTS departamentos;

CREATE TABLE departamentos (
    id_depto  INTEGER PRIMARY KEY AUTOINCREMENT,   -- PK: número único, se incrementa solo
    nombre    TEXT NOT NULL UNIQUE,                -- obligatorio y no repetido
    ubicacion TEXT NOT NULL                        -- piso o local del área
);

CREATE TABLE empleados (
    id_emp    INTEGER PRIMARY KEY AUTOINCREMENT,   -- PK: identificador del empleado
    nombre    TEXT NOT NULL,                       -- nombre completo
    cargo     TEXT NOT NULL,                       -- Analista, Desarrollador, Jefe...
    sueldo    REAL NOT NULL,                       -- sueldo mensual en soles
    email     TEXT UNIQUE,                         -- correo único; puede estar vacío (NULL)
    ciudad    TEXT DEFAULT 'Cusco',                -- si no se indica, se guarda 'Cusco'
    id_depto  INTEGER NOT NULL,                    -- columna que REFERENCIA a departamentos
    FOREIGN KEY (id_depto) REFERENCES departamentos(id_depto)
);
""")
print("Tablas creadas correctamente")

Tablas creadas correctamente


### Insertar datos con `INSERT INTO`

Las columnas `INTEGER PRIMARY KEY AUTOINCREMENT` **no se escriben**: SQLite las genera solo (1, 2, 3...). Fíjate en que **Jose Ttito no tiene correo** (`NULL`): lo usaremos más adelante.

In [3]:
conn.executescript("""
INSERT INTO departamentos (nombre, ubicacion) VALUES
    ('Sistemas',       'Piso 3 - Av. El Sol'),    -- id_depto = 1
    ('Ventas',         'Piso 1 - Av. El Sol'),    -- id_depto = 2
    ('Administracion', 'Piso 2 - Av. El Sol'),    -- id_depto = 3
    ('Soporte',        'Piso 1 - Urb. Ttio'),     -- id_depto = 4
    ('Proyectos',      'Piso 4 - Av. El Sol');    -- id_depto = 5

INSERT INTO empleados (nombre, cargo, sueldo, email, ciudad, id_depto) VALUES
    ('Carlos Quispe', 'Desarrollador',   3500.00, 'cquispe@ayni.pe',  'Cusco', 1),
    ('Lucia Mamani',  'Analista',        2800.00, 'lmamani@ayni.pe',  'Cusco', 1),
    ('Pedro Condori', 'Jefe de Ventas',  4200.00, 'pcondori@ayni.pe', 'Cusco', 2),
    ('Ana Huanca',    'Vendedora',       2200.00, 'ahuanca@ayni.pe',  'Puno',  2),
    ('Jose Ttito',    'Administrador',   3100.00, NULL,               'Cusco', 3),
    ('Maria Ccopa',   'Soporte TI',      2600.00, 'mccopa@ayni.pe',   'Cusco', 4),
    ('Roberto Apaza', 'Desarrollador',   3800.00, 'rapaza@ayni.pe',   'Cusco', 1),
    ('Sofia Huallpa', 'Lider Proyectos', 4500.00, 'shuallpa@ayni.pe', 'Cusco', 5);
""")
print("Datos insertados correctamente")

Datos insertados correctamente


### Comprobar que los datos están

Con `pd.read_sql_query(...)` hacemos nuestro primer `SELECT`: mostrar todo el contenido de una tabla.

In [4]:
pd.read_sql_query("SELECT * FROM departamentos", conn)

,id_depto,nombre,ubicacion
0,1,Sistemas,Piso 3 - Av. El Sol
1,2,Ventas,Piso 1 - Av. El Sol
2,3,Administracion,Piso 2 - Av. El Sol
3,4,Soporte,Piso 1 - Urb. Ttio
4,5,Proyectos,Piso 4 - Av. El Sol


In [5]:
pd.read_sql_query("SELECT * FROM empleados", conn)

,id_emp,nombre,cargo,sueldo,email,ciudad,id_depto
0,1,Carlos Quispe,Desarrollador,3500.0,cquispe@ayni.pe,Cusco,1
1,2,Lucia Mamani,Analista,2800.0,lmamani@ayni.pe,Cusco,1
2,3,Pedro Condori,Jefe de Ventas,4200.0,pcondori@ayni.pe,Cusco,2
3,4,Ana Huanca,Vendedora,2200.0,ahuanca@ayni.pe,Puno,2
4,5,Jose Ttito,Administrador,3100.0,NaN,Cusco,3
5,6,Maria Ccopa,Soporte TI,2600.0,mccopa@ayni.pe,Cusco,4
6,7,Roberto Apaza,Desarrollador,3800.0,rapaza@ayni.pe,Cusco,1
7,8,Sofia Huallpa,Lider Proyectos,4500.0,shuallpa@ayni.pe,Cusco,5


## 4. `SELECT`: leer datos

`SELECT` es la sentencia para **leer** datos. Su estructura básica es:

```sql
SELECT columnas        -- qué columnas quiero ver (o * para todas)
FROM   tabla           -- de qué tabla leo
WHERE  condicion       -- solo las filas que cumplan la condición
ORDER BY columna       -- ordenar el resultado (ASC o DESC)
LIMIT  n;              -- mostrar solo las primeras n filas
```

### 4.1 Todas las columnas con `*`

El asterisco trae **todas** las columnas. Aquí además los ordenamos de la A a la Z por nombre.

In [6]:
pd.read_sql_query("""
SELECT *                 -- el asterisco trae TODAS las columnas
FROM   empleados         -- de la tabla empleados
ORDER BY nombre ASC      -- ordenados de A a Z
""", conn)

,id_emp,nombre,cargo,sueldo,email,ciudad,id_depto
0,4,Ana Huanca,Vendedora,2200.0,ahuanca@ayni.pe,Puno,2
1,1,Carlos Quispe,Desarrollador,3500.0,cquispe@ayni.pe,Cusco,1
2,5,Jose Ttito,Administrador,3100.0,NaN,Cusco,3
3,2,Lucia Mamani,Analista,2800.0,lmamani@ayni.pe,Cusco,1
4,6,Maria Ccopa,Soporte TI,2600.0,mccopa@ayni.pe,Cusco,4
5,3,Pedro Condori,Jefe de Ventas,4200.0,pcondori@ayni.pe,Cusco,2
6,7,Roberto Apaza,Desarrollador,3800.0,rapaza@ayni.pe,Cusco,1
7,8,Sofia Huallpa,Lider Proyectos,4500.0,shuallpa@ayni.pe,Cusco,5


### 4.2 Solo las columnas que necesitas y alias con `AS`

Pedir solo lo necesario hace el resultado más claro. `AS` **renombra** una columna en el resultado (no cambia la tabla).

In [7]:
pd.read_sql_query("""
SELECT nombre AS Empleado,
       cargo  AS Cargo,
       sueldo AS Sueldo_Mensual
FROM   empleados
""", conn)

,Empleado,Cargo,Sueldo_Mensual
0,Carlos Quispe,Desarrollador,3500.0
1,Lucia Mamani,Analista,2800.0
2,Pedro Condori,Jefe de Ventas,4200.0
3,Ana Huanca,Vendedora,2200.0
4,Jose Ttito,Administrador,3100.0
5,Maria Ccopa,Soporte TI,2600.0
6,Roberto Apaza,Desarrollador,3800.0
7,Sofia Huallpa,Lider Proyectos,4500.0


## 5. `WHERE`: filtrar filas

`WHERE` deja pasar **solo las filas que cumplen una condición**.

| Operador | Significado | Ejemplo |
|---|---|---|
| `=` | Igual a | `ciudad = 'Cusco'` |
| `<>` o `!=` | Diferente de | `ciudad <> 'Lima'` |
| `>` `<` | Mayor / menor que | `sueldo > 3000` |
| `>=` `<=` | Mayor o igual / menor o igual | `sueldo >= 2500` |
| `BETWEEN` | Entre dos valores (incluye los extremos) | `sueldo BETWEEN 2000 AND 4000` |
| `LIKE` | Texto con un patrón (`%` = cualquier cantidad de letras) | `nombre LIKE 'Mar%'` |
| `IN (...)` | Dentro de una lista | `ciudad IN ('Cusco', 'Puno')` |
| `IS NULL` | El campo está vacío | `email IS NULL` |
| `AND` | Se cumplen **ambas** condiciones | `ciudad = 'Cusco' AND sueldo > 3000` |
| `OR` | Se cumple **al menos una** | `ciudad = 'Cusco' OR ciudad = 'Puno'` |
| `NOT` | Niega la condición | `NOT ciudad = 'Lima'` |

> Los textos van entre **comillas simples** (`'Cusco'`) y los números sin comillas.

### 5.1 Una condición con `AND`

Empleados que viven en Cusco **y además** ganan más de 3000.

In [8]:
pd.read_sql_query("""
SELECT nombre, cargo, sueldo, ciudad
FROM   empleados
WHERE  ciudad = 'Cusco'      -- solo empleados cuya ciudad ES Cusco
AND    sueldo > 3000         -- Y ADEMÁS cuyo sueldo sea mayor a 3000
ORDER BY sueldo DESC         -- de mayor a menor sueldo
""", conn)

,nombre,cargo,sueldo,ciudad
0,Sofia Huallpa,Lider Proyectos,4500.0,Cusco
1,Pedro Condori,Jefe de Ventas,4200.0,Cusco
2,Roberto Apaza,Desarrollador,3800.0,Cusco
3,Carlos Quispe,Desarrollador,3500.0,Cusco
4,Jose Ttito,Administrador,3100.0,Cusco


### 5.2 Otros filtros útiles

Cada celda responde una pregunta distinta. Lee la pregunta, ejecuta y comprueba que el resultado la responde.

In [9]:
print("¿Qué empleados tienen un nombre que empieza con C?")
pd.read_sql_query("SELECT nombre, cargo FROM empleados WHERE nombre LIKE 'C%'", conn)

¿Qué empleados tienen un nombre que empieza con C?


,nombre,cargo
0,Carlos Quispe,Desarrollador


In [10]:
print("¿Quiénes trabajan en Cusco o en Puno?")
pd.read_sql_query("SELECT nombre, ciudad FROM empleados WHERE ciudad IN ('Cusco', 'Puno')", conn)

¿Quiénes trabajan en Cusco o en Puno?


,nombre,ciudad
0,Carlos Quispe,Cusco
1,Lucia Mamani,Cusco
2,Pedro Condori,Cusco
3,Ana Huanca,Puno
4,Jose Ttito,Cusco
5,Maria Ccopa,Cusco
6,Roberto Apaza,Cusco
7,Sofia Huallpa,Cusco


In [11]:
print("¿Quiénes ganan entre 2500 y 4000?")
pd.read_sql_query("SELECT nombre, sueldo FROM empleados WHERE sueldo BETWEEN 2500 AND 4000 ORDER BY sueldo", conn)

¿Quiénes ganan entre 2500 y 4000?


,nombre,sueldo
0,Maria Ccopa,2600.0
1,Lucia Mamani,2800.0
2,Jose Ttito,3100.0
3,Carlos Quispe,3500.0
4,Roberto Apaza,3800.0


In [12]:
print("¿Quién no tiene correo registrado?")
pd.read_sql_query("SELECT nombre, email FROM empleados WHERE email IS NULL", conn)

¿Quién no tiene correo registrado?


,nombre,email
0,Jose Ttito,None


> **Atención con los vacíos:** para buscar valores vacíos se usa `IS NULL`, **nunca** `= NULL`. Un `NULL` significa «no se sabe» y no es igual a nada, ni siquiera a otro `NULL`.

## 6. `ORDER BY`, `LIMIT` y `DISTINCT`

- `ORDER BY columna ASC` ordena de menor a mayor (es lo normal); `DESC`, de mayor a menor.
- `LIMIT n` muestra **solo las primeras n filas**. Se usa **después** de ordenar: así sabes cuáles son «las primeras».
- `DISTINCT` **elimina repetidos**: muestra cada valor una sola vez.

### 6.1 Los 3 empleados con mayor sueldo

In [13]:
pd.read_sql_query("""
SELECT nombre, cargo, sueldo
FROM   empleados
ORDER BY sueldo DESC     -- primero se ordena...
LIMIT 3                  -- ...y luego se toman solo 3
""", conn)

,nombre,cargo,sueldo
0,Sofia Huallpa,Lider Proyectos,4500.0
1,Pedro Condori,Jefe de Ventas,4200.0
2,Roberto Apaza,Desarrollador,3800.0


### 6.2 Valores distintos

¿En qué ciudades viven los empleados? Sin `DISTINCT` aparecería una fila por empleado.

In [14]:
print("Sin DISTINCT (una fila por empleado):")
pd.read_sql_query("SELECT ciudad FROM empleados", conn)

Sin DISTINCT (una fila por empleado):


,ciudad
0,Cusco
1,Cusco
2,Cusco
3,Puno
4,Cusco
5,Cusco
6,Cusco
7,Cusco


In [15]:
print("Con DISTINCT (cada ciudad una sola vez):")
pd.read_sql_query("SELECT DISTINCT ciudad FROM empleados", conn)

Con DISTINCT (cada ciudad una sola vez):


,ciudad
0,Cusco
1,Puno


## 7. Funciones de resumen: `COUNT`, `SUM`, `AVG`, `MAX`, `MIN`

Calculan **un solo valor** a partir de muchas filas.

| Función | ¿Qué calcula? | Ejemplo |
|---|---|---|
| `COUNT(*)` | Cuenta las **filas** | `COUNT(*)` → cuántos empleados hay |
| `COUNT(columna)` | Cuenta los valores **que no están vacíos** | `COUNT(email)` |
| `SUM(columna)` | Suma los valores | `SUM(sueldo)` → total de la planilla |
| `AVG(columna)` | Promedio | `AVG(sueldo)` |
| `MAX(columna)` | El valor más alto | `MAX(sueldo)` |
| `MIN(columna)` | El valor más bajo | `MIN(sueldo)` |
| `ROUND(valor, n)` | Redondea a `n` decimales | `ROUND(AVG(sueldo), 2)` |

### 7.1 Estadísticas generales de los empleados

In [16]:
pd.read_sql_query("""
SELECT COUNT(*)                AS Total_Empleados,
       ROUND(AVG(sueldo), 2)   AS Sueldo_Promedio,
       MAX(sueldo)             AS Sueldo_Maximo,
       MIN(sueldo)             AS Sueldo_Minimo,
       SUM(sueldo)             AS Planilla_Total
FROM   empleados
""", conn)

,Total_Empleados,Sueldo_Promedio,Sueldo_Maximo,Sueldo_Minimo,Planilla_Total
0,8,3337.5,4500.0,2200.0,26700.0


### 7.2 `COUNT(*)` frente a `COUNT(columna)`

`COUNT(*)` cuenta todas las filas. `COUNT(email)` **ignora los vacíos**. La diferencia entre ambos es exactamente la cantidad de valores `NULL`.

In [17]:
pd.read_sql_query("""
SELECT COUNT(*)     AS Filas,
       COUNT(email) AS Con_Email
FROM   empleados
""", conn)

,Filas,Con_Email
0,8,7


### 7.3 Funciones de resumen con `WHERE`

Primero `WHERE` filtra las filas y **después** la función resume solo esas filas. Ejemplo: sueldo promedio solo de quienes viven en Cusco.

In [18]:
pd.read_sql_query("""
SELECT COUNT(*)              AS Empleados_Cusco,
       ROUND(AVG(sueldo), 2) AS Sueldo_Promedio_Cusco
FROM   empleados
WHERE  ciudad = 'Cusco'
""", conn)

,Empleados_Cusco,Sueldo_Promedio_Cusco
0,7,3500.0


## 8. Vistazo a `UPDATE` y `DELETE` (DML)

Estas sentencias **cambian** los datos. Son poderosas y peligrosas: sin `WHERE` afectan a **todas** las filas. Haz siempre este chequeo: *primero un `SELECT` con el mismo `WHERE`, después el cambio*.

Vamos a subir el sueldo de Ana Huanca y luego a dejarlo como estaba.

In [19]:
print("ANTES:")
pd.read_sql_query("SELECT nombre, sueldo FROM empleados WHERE nombre = 'Ana Huanca'", conn)

ANTES:


,nombre,sueldo
0,Ana Huanca,2200.0


In [20]:
conn.executescript("UPDATE empleados SET sueldo = 2400 WHERE nombre = 'Ana Huanca'")
print("DESPUÉS DEL UPDATE:")
pd.read_sql_query("SELECT nombre, sueldo FROM empleados WHERE nombre = 'Ana Huanca'", conn)

DESPUÉS DEL UPDATE:


,nombre,sueldo
0,Ana Huanca,2400.0


In [21]:
conn.executescript("UPDATE empleados SET sueldo = 2200 WHERE nombre = 'Ana Huanca'")   # lo dejamos como estaba
print("RESTAURADO:")
pd.read_sql_query("SELECT nombre, sueldo FROM empleados WHERE nombre = 'Ana Huanca'", conn)

RESTAURADO:


,nombre,sueldo
0,Ana Huanca,2200.0


## 9. Resumen del nivel básico

### Comandos aprendidos

| Comando | ¿Para qué sirve? | Ejemplo |
|---|---|---|
| `CREATE TABLE` | Define la estructura de una tabla nueva | `CREATE TABLE clientes (...)` |
| `INSERT INTO` | Agrega filas a una tabla | `INSERT INTO t (a, b) VALUES (1, 'x')` |
| `SELECT ... FROM` | Lee datos de una tabla | `SELECT * FROM empleados` |
| `WHERE` | Filtra filas según una condición | `WHERE ciudad = 'Cusco'` |
| `ORDER BY` | Ordena el resultado (`ASC` o `DESC`) | `ORDER BY sueldo DESC` |
| `LIMIT n` | Muestra solo las primeras `n` filas | `LIMIT 5` |
| `DISTINCT` | Elimina valores repetidos | `SELECT DISTINCT ciudad` |
| `COUNT` `SUM` `AVG` | Cuentan, suman o promedian | `COUNT(*)`, `SUM(sueldo)` |
| `MAX` `MIN` | Valor más alto o más bajo | `MAX(sueldo)` |
| `UPDATE ... SET` | Modifica datos existentes | `UPDATE t SET a = 1 WHERE id = 1` |
| `DELETE FROM` | Elimina filas | `DELETE FROM t WHERE id = 1` |

### Orden de las cláusulas en un `SELECT` (nivel básico)

```
SELECT   (qué columnas mostrar)
FROM     (de qué tabla)
WHERE    (filtrar filas)
ORDER BY (ordenar el resultado)
LIMIT    (limitar la cantidad de filas)
```

En el **nivel intermedio** se añaden `JOIN`, `GROUP BY` y `HAVING`.

## 10. Ejercicios

Escribe tu consulta en una celda nueva con `pd.read_sql_query("""...""", conn)`. Cada ejercicio indica cuántas filas debe devolver para que compruebes tu resultado. La solución está escondida: haz clic en **Ver solución** solo después de intentarlo.

---

**Ejercicio 1.** Muestra el nombre, cargo y sueldo de los empleados del departamento **1** (Sistemas) que ganan **3500 o más**. *Debe devolver 2 filas.*

<details><summary>Ver solución</summary>

```python
pd.read_sql_query("""
SELECT nombre, cargo, sueldo
FROM   empleados
WHERE  id_depto = 1 AND sueldo >= 3500
""", conn)
```
</details>

---

**Ejercicio 2.** Muestra los **2 empleados con menor sueldo** (nombre y sueldo). *Debe devolver 2 filas: Ana Huanca y Maria Ccopa.*

<details><summary>Ver solución</summary>

```python
pd.read_sql_query("""
SELECT nombre, sueldo
FROM   empleados
ORDER BY sueldo ASC
LIMIT 2
""", conn)
```
</details>

---

**Ejercicio 3.** ¿Cuántos **cargos distintos** hay en la empresa? Muéstralos. *Debe devolver 7 filas.*

<details><summary>Ver solución</summary>

```python
pd.read_sql_query("SELECT DISTINCT cargo FROM empleados", conn)
```
</details>

---

**Ejercicio 4.** ¿Cuántos empleados **no viven en Cusco**? *Debe devolver el número 1.*

<details><summary>Ver solución</summary>

```python
pd.read_sql_query("SELECT COUNT(*) AS No_Viven_En_Cusco FROM empleados WHERE ciudad <> 'Cusco'", conn)
```
</details>

---

**Ejercicio 5.** ¿Cuál es el **sueldo promedio** de los empleados de Cusco? *Debe devolver 3500.0.*

<details><summary>Ver solución</summary>

```python
pd.read_sql_query("SELECT AVG(sueldo) AS Sueldo_Promedio FROM empleados WHERE ciudad = 'Cusco'", conn)
```
</details>

---

**Ejercicio 6.** ¿Quién **no tiene correo** registrado? Muestra su nombre y cargo. *Debe devolver 1 fila: Jose Ttito.*

<details><summary>Ver solución</summary>

```python
pd.read_sql_query("SELECT nombre, cargo FROM empleados WHERE email IS NULL", conn)
```
</details>

---

*SQLite Básico · TECSUP · Fundamentos de Gestión de Datos · Docente: Pilar Rocío Sayán Mejía*